# Part 2 · How far L4 goes: identity authorisation without the L7 hop

**The story for the customer:** in ambient, ztunnel enforces access policy at L4 on every hop, using each workload's certificate identity, with **no proxy in the request path**. Before you reach for an L7 waypoint, it is worth seeing how much you can already do here: authorise on identity, on namespace, on conditions, deny selectively, and even tell apart two pods that share a ServiceAccount. Every request you keep at L4 is an L7 hop you avoid.

This part runs on `mesh1` only and needs no peering, so it stands alone. We take one app, a petshop, through the L4 model end to end. Part 3 then adds the waypoint for the things that genuinely need L7.

The trust domain on `mesh1` is **`mesh1`** (a Helm value at install, not `cluster.local`), so identities read `spiffe://mesh1/ns/petshop/sa/<sa>`.

**The cast:**

| Workload | ServiceAccount → identity | Role |
|---|---|---|
| `petstore` | `sa/petstore` | the API — `GET /pets`, `DELETE /pets/{id}` |
| `storefront` | `sa/storefront` | client the L4 policy **allows** |
| `analytics` | `sa/analytics` | client the L4 policy **denies** |
| `checkout-blue` | `sa/checkout` | shares one SA with green … |
| `checkout-green` | `sa/checkout` | … so both present the **same** SVID |

Each client curls `petstore:8080/pets` every 2 seconds and prints the HTTP code. The observe cells read the client's last log line, which is why each policy step sleeps briefly first. A denied client shows `000` (connection reset), an allowed one `200`.

> **Kernel:** Bash (Select Kernel → Jupyter Kernel → **Bash**).
> This notebook is **self-contained**: run **Connect**, then **Reset**, then the steps top to bottom.
> It needs `./setup.sh` to have been run once. After a laptop sleep: `./demo-scripts/wake.sh`.

### Connect

Sets the contexts, the Solo `istioctl` build and your licence env, and confirms the platform is up. **Run this first, always.**

In [ ]:
# Connect — context, image/chart versions, licences. Run this first, always.
[ -d istio-ambient-demo-kind ] && cd istio-ambient-demo-kind || :
export CTX=kind-mesh1 ISTIO_NS=istio-system TD=mesh1
export ISTIOCTL=$HOME/.istioctl/bin/istioctl-1.30.3-solo
export HUB=us-docker.pkg.dev/soloio-img/istio TAG=1.30.3-solo
export HREPO=oci://us-docker.pkg.dev/soloio-img/istio-helm HVER=1.30.3-solo
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "context: $CTX ; trust domain: $TD ; licence: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"
kubectl --context $CTX -n $ISTIO_NS get ds ztunnel >/dev/null 2>&1 \
  && echo "ambient mesh: up on ${CTX#kind-}" \
  || echo "mesh not found on ${CTX#kind-} — run ./setup.sh (or ./demo-scripts/wake.sh after a sleep)"

## Reset · run this to (re)start the demo

Clears the L4 policies, the `warehouse` namespace and the tier annotations, and reverts ztunnel to claims-off if a previous run turned it on. Safe on a fresh cluster: every command is a no-op when there is nothing to remove.

In [ ]:
# Reset — clean slate for this demo. Safe to run on a fresh cluster (all no-ops).
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" \
  "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" \
  "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX -n petshop delete authorizationpolicy \
  allow-storefront allow-checkout allow-gold-checkout l4-allow-petshop-namespace l4-deny-analytics \
  --ignore-not-found 2>/dev/null || true
kubectl --context $CTX delete ns warehouse --ignore-not-found 2>/dev/null || true
for d in checkout-blue checkout-green; do
  kubectl --context $CTX -n petshop patch deploy $d --type=json \
    -p='[{"op":"remove","path":"/spec/template/metadata/annotations/solo.io.security-claims~1tier"}]' 2>/dev/null || true
done
# ztunnel back to claims-off, but only if a previous run turned it on
if kubectl --context $CTX -n $ISTIO_NS get ds ztunnel \
     -o jsonpath='{.spec.template.spec.containers[0].env[?(@.name=="ENABLE_WORKLOAD_CLAIMS")].value}' 2>/dev/null | grep -q true; then
  echo "  reverting ztunnel to claims-off …"
  helm --kube-context $CTX upgrade -i ztunnel $HREPO/ztunnel -n $ISTIO_NS --version $HVER --wait -f - >/dev/null <<EOF
profile: ambient
hub: $HUB
tag: $TAG
namespace: $ISTIO_NS
istioNamespace: $ISTIO_NS
multiCluster:
  clusterName: mesh1
network: mesh1
platforms:
  peering:
    enabled: true
env:
  LOG_FORMAT: json
  L7_ENABLED: "true"
  SKIP_VALIDATE_TRUST_DOMAIN: "true"
EOF
  kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/ztunnel --timeout=180s
fi
echo "  ✓ clean — §2.1 below (re)deploys the petshop; every step is idempotent"

## 2.1 · Deploy the petshop app

**What we're doing:** deploy the petshop (an API plus four client workloads) into one namespace and enrol it in the mesh.

**How:** a single `istio.io/dataplane-mode=ambient` label on the namespace, no restarts and no sidecars.

**What you'll see:** the five pods running, ready for the identity walk-through.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: petshop
  labels:
    istio.io/dataplane-mode: ambient
EOF

kubectl --context $CTX apply -f yaml/10-petshop/
kubectl --context $CTX -n petshop rollout status deploy/petstore deploy/storefront deploy/analytics deploy/checkout-blue deploy/checkout-green --timeout=180s
kubectl --context $CTX -n petshop get pods

## 2.2 · The identity is the certificate

**What we're doing:** show that a workload's identity is its mTLS certificate, issued from its ServiceAccount, and that this is what policy matches on.

**How:** list the leaf certificates ztunnel holds, next to the pods.

**What you'll see:** five pods but only **four** certificates. `checkout-blue` and `checkout-green` never appear by name because they share `sa/checkout`, so ztunnel presents one certificate for both. To the mesh they are the same caller. That is the ceiling §2.5 runs into and §2.7 removes.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# five pods, but their identity is the ServiceAccount…
kubectl --context $CTX -n petshop get pods \
  -o custom-columns=POD:.metadata.name,SERVICEACCOUNT:.spec.serviceAccountName

# …so the ztunnel on their node holds only FOUR leaf certs — checkout appears once
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
$ISTIOCTL --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE NAME|petshop" | grep -Ei "name|leaf"

## 2.3 · Authorise on identity, at L4

**What we're doing:** allow only the `storefront` workload to reach `petstore`, and let ztunnel deny everything else, with no waypoint and no app change.

**How:** one `AuthorizationPolicy` that selects `petstore` and permits the `storefront` identity. ztunnel fails closed: once any ALLOW selects a workload, everything not named is denied. Principals use the `mesh1` trust domain, so a `cluster.local/...` principal here would match nothing.

**What you'll see:** `storefront` gets `200`, while `analytics` and both `checkout` pods get `000` (connection refused), decided purely on identity.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-storefront, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["mesh1/ns/petshop/sa/storefront"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
# each client curls petstore every 2s; read its latest result and show the verdict
echo "  ══ who can reach petstore?  (only the storefront identity is allowed) ══"
for d in storefront analytics checkout-blue checkout-green; do
  code=$(kubectl --context $CTX -n petshop logs deploy/$d --tail=1 2>/dev/null | awk '{print $NF}')
  [ "$code" = "200" ] && v="200      ✓ ALLOW" || v="${code:-000}   ✗ DENY"
  printf "    %-16s %s\n" "$d" "$v"
done

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 700 260" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="700" height="260" rx="10" fill="#f8fafc"/> <text x="350" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">ztunnel decides on identity: allow storefront, deny the rest</text> <defs> <marker id="lg" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker> <marker id="lr" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker> </defs> <!-- callers on the left --> <rect x="24" y="52" width="150" height="34" rx="7" fill="#dcfce7" stroke="#16a34a"/> <text x="99" y="74" text-anchor="middle" font-size="11.5" fill="#14532d">storefront</text> <rect x="24" y="98" width="150" height="34" rx="7" fill="#fee2e2" stroke="#dc2626"/> <text x="99" y="120" text-anchor="middle" font-size="11.5" fill="#7f1d1d">analytics</text> <rect x="24" y="144" width="150" height="34" rx="7" fill="#fee2e2" stroke="#dc2626"/> <text x="99" y="166" text-anchor="middle" font-size="11.5" fill="#7f1d1d">checkout-blue</text> <rect x="24" y="190" width="150" height="34" rx="7" fill="#fee2e2" stroke="#dc2626"/> <text x="99" y="212" text-anchor="middle" font-size="11.5" fill="#7f1d1d">checkout-green</text> <!-- ztunnel gate in the middle --> <rect x="300" y="86" width="90" height="90" rx="10" fill="#e0e7ff" stroke="#6366f1" stroke-width="2"/> <text x="345" y="126" text-anchor="middle" font-size="12" font-weight="700" fill="#312e81">ztunnel</text> <text x="345" y="142" text-anchor="middle" font-size="9.5" fill="#4338ca">L4 authz</text> <!-- arrows callers -> ztunnel --> <line x1="174" y1="69"  x2="300" y2="110" stroke="#16a34a" stroke-width="2" marker-end="url(#lg)"/> <line x1="174" y1="115" x2="300" y2="128" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#lr)"/> <line x1="174" y1="161" x2="300" y2="140" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#lr)"/> <line x1="174" y1="207" x2="300" y2="158" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#lr)"/> <!-- petstore on the right, only the allowed one gets through --> <line x1="390" y1="120" x2="500" y2="120" stroke="#16a34a" stroke-width="2" marker-end="url(#lg)"/> <text x="445" y="112" text-anchor="middle" font-size="10" fill="#166534">200</text> <rect x="500" y="98" width="176" height="44" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/> <text x="588" y="125" text-anchor="middle" font-size="12.5" font-weight="600" fill="#1e293b">petstore</text> <text x="350" y="246" text-anchor="middle" font-size="11" fill="#64748b">One ALLOW on the storefront identity. ztunnel fails closed: everything unnamed is denied, no waypoint.</text> </svg></div>

**Watching in the Gloo UI and the denied workloads still show edges to `petstore`?** The graph is drawn from **completed-request telemetry**. A workload denied at L4 never completes a request, so it should draw nothing, but a pre-policy 200 can linger as a stale edge. Turn **"Idle Nodes" OFF** (Graph Settings) so only live traffic draws, and set Traffic to **Last 1 min**. The definitive, per-connection allow/deny verdicts are in ztunnel's access logs, next.

## 2.4 · Read the L4 access logs

**What we're doing:** show that the allow/deny decisions are auditable, tagged with the caller's identity, at L4 with no waypoint.

**How:** read ztunnel's access log, which records every connection with the peer SPIFFE identities and the outcome.

**What you'll see:** `storefront` as **ALLOW** and `analytics` as **DENY**, each line carrying its `src.identity`.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
kubectl --context $CTX -n $ISTIO_NS logs "$ZT" --tail=400 \
 | grep '"scope":"access"' | grep petshop | tail -8 \
 | jq -rc '{src:(.["src.identity"]//"-"),
            dst:(.["dst.service"]//.["dst.identity"]//"-"),
            dir:(.direction//"-"),
            result:(if (.error//"")=="" then "ALLOW" else ("DENY: "+(.error|tostring)) end)}'

## 2.5 · The limit of service-account identity

**What we're doing:** show the one thing this model cannot do: tell apart two pods that share a ServiceAccount.

**How:** add `sa/checkout` to the allowed set and call from both checkout pods.

**What you'll see:** both `checkout-blue` and `checkout-green` get in, and there is no L4 rule that admits one and blocks the other. They share `sa/checkout`, present the same certificate, and to ztunnel they are the same caller.

Why it matters for the customer: the everyday version is worse than blue/green. Any pod that never sets a ServiceAccount runs as the namespace `default`, so an ALLOW written for "the payments app" quietly admits every workload in that namespace. The two fixes are to give every workload its own ServiceAccount (as this app does) and, where pods genuinely share one, to close the gap with **workload claims** in §2.7.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-checkout, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["mesh1/ns/petshop/sa/checkout"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
echo "  ══ now sa/checkout is allowed too  ·  BOTH checkout pods get in ══"
for d in storefront analytics checkout-blue checkout-green; do
  code=$(kubectl --context $CTX -n petshop logs deploy/$d --tail=1 2>/dev/null | awk '{print $NF}')
  [ "$code" = "200" ] && v="200      ✓ ALLOW" || v="${code:-000}   ✗ DENY"
  printf "    %-16s %s\n" "$d" "$v"
done
echo "    checkout-blue and checkout-green share sa/checkout — no L4 rule separates them"

## 2.6 · What else L4 can decide: namespace, conditions, DENY

**What we're doing:** show that identity is not the only thing ztunnel can authorise on. It also decides on source namespace, source IP, destination port and SNI, directly or in a CEL `when` clause, and a `DENY` always beats an `ALLOW`.

**How:** add a second caller in its own `warehouse` namespace, then walk it through allow, block and an explicit deny.

**What you'll see:** the warehouse caller allowed when the policy admits its namespace, then blocked when we narrow it, and finally an explicit `DENY` overriding an `ALLOW`. You can watch it appear and disappear in the Gloo UI Graph.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# A second client, in ANOTHER namespace — its identity is spiffe://mesh1/ns/warehouse/sa/warehouse-svc
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: warehouse
  labels:
    istio.io/dataplane-mode: ambient
---
apiVersion: v1
kind: ServiceAccount
metadata: { name: warehouse-svc, namespace: warehouse }
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: warehouse-svc, namespace: warehouse, labels: { app: warehouse-svc } }
spec:
  replicas: 1
  selector: { matchLabels: { app: warehouse-svc } }
  template:
    metadata: { labels: { app: warehouse-svc } }
    spec:
      serviceAccountName: warehouse-svc
      containers:
        - name: client
          image: curlimages/curl:8.14.1
          command: ["/bin/sh","-c"]
          args:
            - |
              while true; do
                code=$(curl -s -o /dev/null -w '%{http_code}' --max-time 3 http://petstore.petshop:8080/pets || echo 000)
                echo "$(date -u +%H:%M:%S) warehouse-svc -> GET petstore.petshop/pets : $code"
                sleep 2
              done
EOF
kubectl --context $CTX -n warehouse rollout status deploy/warehouse-svc --timeout=90s

# ── verify: warehouse client is up and its identity is in the warehouse trust domain ──
kubectl --context $CTX -n warehouse get pods
echo "no ALLOW admits it yet, so it is denied for now:"
sleep 6; kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# ONE ALLOW admits callers from TWO namespaces — decided by a CEL `when` on source.namespace
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-allow-petshop-namespace, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - to:   [{ operation: { ports: ["8080"] } }]
    when: [{ key: source.namespace, values: ["petshop", "warehouse"] }]
EOF
sleep 14
echo "petshop:   $(kubectl --context $CTX -n petshop   logs deploy/storefront    --tail=1)"
echo "warehouse: $(kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1)"
# both 200 — refresh the Graph: warehouse appears, serving petstore

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# Now BLOCK it — same policy, one value removed. warehouse fails closed.
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-allow-petshop-namespace, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - to:   [{ operation: { ports: ["8080"] } }]
    when: [{ key: source.namespace, values: ["petshop"] }]
EOF
sleep 14
echo "warehouse: $(kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1)"

# the deny as ztunnel logged it — FAIL-CLOSED: allow policies exist, none matched
for i in $(seq 1 12); do
  LINE=$(kubectl --context $CTX -n $ISTIO_NS logs -l app=ztunnel --tail=600 2>/dev/null | grep "policy rejection" | grep warehouse | tail -1)
  [ -n "$LINE" ] && break; sleep 5
done
echo "$LINE" | python3 -c 'import sys,json; l=sys.stdin.read().strip(); d=json.loads(l) if l else {}; print(d.get("src.identity","(no deny line found — the policy may still be converging; re-run this cell)"),"->",d.get("dst.service",""),"\n  error:",d.get("error",""))'

**DENY beats ALLOW.** ztunnel evaluates CUSTOM → DENY → ALLOW, so a `DENY` rule overrides the namespace `ALLOW`. Here `analytics` is in `petshop` (the ALLOW admits it) but a `DENY` on its identity blocks it anyway — a clean carve-out with no change to the broader policy. From the client both denials are the same `000` — the difference is in ztunnel's log: the warehouse deny read `allow policies exist, but none allowed` (fail-closed); this one reads `explicitly denied by: petshop/l4-deny-analytics` — the log **names the policy** that fired.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-deny-analytics, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: DENY
  rules:
  - from: [{ source: { principals: ["mesh1/ns/petshop/sa/analytics"] } }]
EOF
sleep 14
for d in storefront analytics; do echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"; done

for i in $(seq 1 12); do
  LINE=$(kubectl --context $CTX -n $ISTIO_NS logs -l app=ztunnel --tail=600 2>/dev/null | grep "policy rejection" | grep analytics | tail -1)
  [ -n "$LINE" ] && break; sleep 5
done
echo "$LINE" | python3 -c 'import sys,json; l=sys.stdin.read().strip(); d=json.loads(l) if l else {}; print(d.get("src.identity","(no deny line found — the policy may still be converging; re-run this cell)"),"->",d.get("dst.service",""),"\n  error:",d.get("error",""))'

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# reset the L4-surface policies before closing the gap
kubectl --context $CTX -n petshop delete authorizationpolicy l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found

# ── verify: only the SA-wide allow-storefront + allow-checkout remain ──
kubectl --context $CTX -n petshop get authorizationpolicy

## 2.7 · Close the gap with workload claims

**What we're doing:** solve the §2.5 problem, telling apart two pods on one ServiceAccount, still at L4 and still with no waypoint.

**How:** turn on `ENABLE_WORKLOAD_CLAIMS`. ztunnel then requests a certificate **per pod** instead of per ServiceAccount, istiod embeds signed claims in each one at issuance, and an `AuthorizationPolicy` matches those claims with CEL. On the 1.30 line this is one Helm value on ztunnel, same chart and version, all the multicluster values kept.

**What you'll see (over the next few cells):** every pod now holding its own certificate, `checkout-blue` carrying `tier: gold` and `checkout-green` carrying `tier: silver` in that certificate, and a claims-based policy that finally lets blue in and keeps green out.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# ONE new value: ENABLE_WORKLOAD_CLAIMS. Everything else identical to the standup.
helm --kube-context $CTX upgrade -i ztunnel $HREPO/ztunnel -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
hub: $HUB
tag: $TAG
namespace: $ISTIO_NS
istioNamespace: $ISTIO_NS
multiCluster:
  clusterName: mesh1
network: mesh1
platforms:
  peering:
    enabled: true
env:
  LOG_FORMAT: json
  L7_ENABLED: "true"
  SKIP_VALIDATE_TRUST_DOMAIN: "true"
  ENABLE_WORKLOAD_CLAIMS: "true"    # per-POD certs + claim enforcement
EOF
kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/ztunnel --timeout=180s

# ── verify: the flag is live on ztunnel ──
kubectl --context $CTX -n $ISTIO_NS get ds ztunnel \
  -o jsonpath='{.spec.template.spec.containers[0].env[?(@.name=="ENABLE_WORKLOAD_CLAIMS")].name}={.spec.template.spec.containers[0].env[?(@.name=="ENABLE_WORKLOAD_CLAIMS")].value}{"\n"}'

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# every workload now holds a PER-POD cert (contrast §2.2: one per ServiceAccount)
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
$ISTIOCTL --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE|checkout"

**Annotate the pod — the claim is embedded in its certificate at issuance.** istiod reads `solo.io.security-claims/<key>` off the pod and bakes it into the cert, alongside auto claims for the workload name, namespace and pod. The SPIFFE URI never changes (still `sa/checkout`); the claims ride alongside it.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# blue is the gold tier, green is silver — one annotation each, pods roll
kubectl --context $CTX -n petshop patch deploy checkout-blue  -p '{"spec":{"template":{"metadata":{"annotations":{"solo.io.security-claims/tier":"gold"}}}}}'
kubectl --context $CTX -n petshop patch deploy checkout-green -p '{"spec":{"template":{"metadata":{"annotations":{"solo.io.security-claims/tier":"silver"}}}}}'
kubectl --context $CTX -n petshop rollout status deploy/checkout-blue deploy/checkout-green --timeout=120s
sleep 5

# ── verify: the tier annotation is on each pod (istiod bakes it into the cert) ──
kubectl --context $CTX -n petshop get pod -l app=checkout \
  -o custom-columns=POD:.metadata.name,TIER:'.metadata.annotations.solo\.io\.security-claims/tier'

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
# open the certs with openssl and read the claims off the SAN. Each cert gains an
# otherName SAN under Solo's OID (1.3.6.1.4.1.65865.1.1) whose payload is base64url JSON.
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=mesh1-worker -o jsonpath='{.items[0].metadata.name}')
$ISTIOCTL --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" -o json > /tmp/zt-certs.json

for pod in checkout-blue checkout-green; do
  jq -r ".[] | select(.identity | contains(\"$pod\")) | .certChain[0].pem" /tmp/zt-certs.json \
    | base64 -d > /tmp/$pod.pem
  echo "── $pod ──────────────────────────────────────────────"
  openssl x509 -in /tmp/$pod.pem -noout -text | grep -A1 "Subject Alternative Name" | tail -1
  openssl x509 -in /tmp/$pod.pem -noout -text | grep -o '65865\.1\.1:.*' | cut -d: -f2 \
    | python3 -c 'import sys,base64,json; p=sys.stdin.read().strip(); print(json.dumps(json.loads(base64.urlsafe_b64decode(p+"="*(-len(p)%4))), indent=2))'
done

Same URI SAN on both (`…/sa/checkout`) — but blue's cert says `tier: gold` and green's says `tier: silver`, signed by istiod. Now authorize on it: swap the SA-wide checkout ALLOW for a claims-scoped one — the same `when` CEL shape as §2.6, over `source.claims` instead of `source.namespace` (the `/` in the annotation key becomes `.` in the claim key). Fail-closed: a workload with no claim never matches the ALLOW.

In [ ]:
: "${CTX:=kind-mesh1}" "${ISTIO_NS:=istio-system}" "${TD:=mesh1}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}" "${HUB:=us-docker.pkg.dev/soloio-img/istio}" "${TAG:=1.30.3-solo}" "${HREPO:=oci://us-docker.pkg.dev/soloio-img/istio-helm}" "${HVER:=1.30.3-solo}"
kubectl --context $CTX -n petshop delete authorizationpolicy allow-checkout l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata:
  name: allow-gold-checkout
  namespace: petshop
spec:
  selector:
    matchLabels: { app: petstore }
  action: ALLOW
  rules:
    - from:
        - source:
            principals:
              - mesh1/ns/petshop/sa/checkout
      to:
        - operation:
            ports: ["8080"]
      when:
        - key: "source.claims['solo.io.security-claims.tier']"
          values: ["gold"]
EOF
kubectl --context $CTX -n petshop get authorizationpolicy allow-gold-checkout -o jsonpath='{.status.conditions[0].message}'; echo; echo

# fresh per-pod certs + the new policy take ~20-30s to converge
for i in $(seq 1 24); do
  B=$(kubectl --context $CTX -n petshop exec deploy/checkout-blue -- sh -c \
        'curl -s -o /dev/null -w "%{http_code}" --max-time 3 http://petstore:8080/' 2>/dev/null)
  G=$(kubectl --context $CTX -n petshop exec deploy/checkout-green -- sh -c \
        'curl -s -o /dev/null -w "%{http_code}" --max-time 3 http://petstore:8080/' 2>/dev/null)
  echo "  waiting for convergence ($i/24): blue=$B green=$G (want blue=200, green!=200)"
  [ "$B" = "200" ] && [ "$G" != "200" ] && break; sleep 5
done
if [ "$G" = "200" ]; then
  echo "!! green is still allowed — is another ALLOW policy admitting it? kubectl -n petshop get authorizationpolicy"
fi
echo
echo "checkout-blue  (tier gold)   -> $B"
echo "checkout-green (tier silver) -> ${G:-DENIED}   (same sa/checkout, told apart by its cert claim)"

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 700 250" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="700" height="250" rx="10" fill="#f8fafc"/> <text x="350" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Workload claims: same ServiceAccount, told apart by a signed claim</text> <defs> <marker id="cg" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker> <marker id="cr" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker> </defs> <!-- blue pod (gold) --> <rect x="24" y="58" width="210" height="52" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/> <text x="129" y="79" text-anchor="middle" font-size="12" font-weight="600" fill="#14532d">checkout-blue</text> <text x="129" y="96" text-anchor="middle" font-size="10" fill="#166534">sa/checkout · cert claim tier=gold</text> <!-- green pod (silver) --> <rect x="24" y="140" width="210" height="52" rx="8" fill="#fee2e2" stroke="#dc2626" stroke-width="1.5"/> <text x="129" y="161" text-anchor="middle" font-size="12" font-weight="600" fill="#7f1d1d">checkout-green</text> <text x="129" y="178" text-anchor="middle" font-size="10" fill="#991b1b">sa/checkout · cert claim tier=silver</text> <!-- policy gate --> <rect x="300" y="88" width="110" height="74" rx="10" fill="#e0e7ff" stroke="#6366f1" stroke-width="2"/> <text x="355" y="120" text-anchor="middle" font-size="11" font-weight="700" fill="#312e81">CEL when</text> <text x="355" y="137" text-anchor="middle" font-size="9.5" fill="#4338ca">tier == "gold"</text> <line x1="234" y1="84"  x2="300" y2="112" stroke="#16a34a" stroke-width="2" marker-end="url(#cg)"/> <line x1="234" y1="166" x2="300" y2="138" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#cr)"/> <!-- petstore --> <line x1="410" y1="125" x2="500" y2="125" stroke="#16a34a" stroke-width="2" marker-end="url(#cg)"/> <text x="455" y="117" text-anchor="middle" font-size="10" fill="#166534">200</text> <rect x="500" y="103" width="176" height="44" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/> <text x="588" y="130" text-anchor="middle" font-size="12.5" font-weight="600" fill="#1e293b">petstore</text> <text x="350" y="228" text-anchor="middle" font-size="11" fill="#64748b">Both share sa/checkout, so §2.5 could not separate them. The per-pod cert claim can, still at L4.</text> </svg></div>

`checkout-blue -> 200`, `checkout-green -> denied`, same ServiceAccount, told apart at L4 by the workload's own certificate. That is the exact thing §2.5 could not do.

That closes out the L4 story. Everything so far, identity, namespace, conditions, a deny, and now per-pod claims, was enforced by ztunnel at L4 with **no proxy in the path**. Part 3 picks up the things that genuinely need L7.